# GANForge -- GameGAN (Standalone Notebook)

**Game environment / next-frame simulation**

This is a **self-contained extract** of the GameGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="12"></a>
## 12. GameGAN -- Game environment / next-frame simulation

### 12.1 Problem Statement & Why this GAN

Kim et al. (2020, NVIDIA) asked whether a neural network could **learn to
simulate a video game's engine purely by watching gameplay** -- given the
current frame and the player's action, predict the next frame, without
ever being given the game's actual rules/source code. This is a
fundamentally different problem from every earlier section: the generator
is not conditioned on noise-plus-a-label but on **the previous visual
state plus a discrete control input**, and must implicitly learn
game-specific dynamics (walls block movement, the agent moves in the
requested direction) purely from pixel-level supervision. Full GameGAN
additionally has a **memory module** (an external, addressable memory
bank for long-term spatial consistency -- "remember what's off-screen")
and separates a **dynamics engine** (predicts what changes) from a
**rendering engine** (turns that into pixels).

⚠️ **Warning -- Simplification.** Real GameGAN was trained on thousands of
hours of actual Pac-Man gameplay with a full external memory module. We
implement its **core mechanism** -- action-conditioned next-frame
prediction with a small recurrent "dynamics" state standing in for the
external memory -- on a **custom GridWorld** we can simulate instantly and
offline, which is why no pretrained-weight-loading subsection appears
here (no broadly available, license-clear pretrained GameGAN checkpoint
exists to load).

### 12.2 Data Collection & Preparation

In [ ]:
GAMEGAN_GRID_SIZE = 8
GAMEGAN_IMG_SIZE = 32
GAMEGAN_N_ACTIONS = 5
GAMEGAN_ACTION_NAMES = ["up", "down", "left", "right", "stay"]
GAMEGAN_BATCH_SIZE = 64
GAMEGAN_EPOCHS = 10

class GridWorld:
    '''A tiny, fully-deterministic "game": an agent (red square) moves on an
    8x8 grid with a fixed wall pattern (dark squares), rendered as a 32x32
    RGB image. This IS the "game engine" GameGAN has to learn implicitly
    from pixels alone -- we only use it here to generate training/ground
    -truth rollouts and to compare predictions against.'''
    ACTION_DELTAS = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0), 4: (0, 0)}  # up, down, left, right, stay

    def __init__(self, grid_size=GAMEGAN_GRID_SIZE, img_size=GAMEGAN_IMG_SIZE, rng=None):
        self.grid_size, self.img_size = grid_size, img_size
        self.rng = rng or np.random.default_rng()
        self.walls = {(i, 4) for i in range(2, 6)}   # a fixed obstacle wall -- the "rule" the model must implicitly discover
        self.reset()

    def reset(self):
        while True:
            pos = (int(self.rng.integers(0, self.grid_size)), int(self.rng.integers(0, self.grid_size)))
            if pos not in self.walls:
                self.agent_pos = pos
                return self.render()

    def step(self, action: int):
        dx, dy = self.ACTION_DELTAS[action]
        x, y = self.agent_pos
        nx = int(np.clip(x + dx, 0, self.grid_size - 1))
        ny = int(np.clip(y + dy, 0, self.grid_size - 1))
        if (nx, ny) not in self.walls:   # walls block movement -- this is the "physics" GameGAN must learn purely from pixels
            self.agent_pos = (nx, ny)
        return self.render()

    def render(self) -> torch.Tensor:
        cell = self.img_size // self.grid_size
        img = np.ones((self.img_size, self.img_size, 3), dtype=np.float32) * 0.9
        for wx, wy in self.walls:
            img[wy * cell:(wy + 1) * cell, wx * cell:(wx + 1) * cell] = [0.2, 0.2, 0.2]
        ax, ay = self.agent_pos
        img[ay * cell:(ay + 1) * cell, ax * cell:(ax + 1) * cell] = [0.9, 0.15, 0.15]
        return torch.from_numpy(img).permute(2, 0, 1) * 2 - 1

class GameGANDataset(Dataset):
    '''Random-action rollouts through GridWorld, cut into independent
    (frame_t, action_t, frame_t+1) TRAINING TRIPLES. Unlike CycleGAN
    (Section 8), this IS paired data -- the simulator gives us exact
    ground-truth next-frames, so we can (and do) use a direct pixel
    reconstruction loss alongside the adversarial one.'''
    def __init__(self, n_episodes=250, episode_len=16, seed=SEED):
        rng = np.random.default_rng(seed)
        self.triples = []
        for _ in range(n_episodes):
            env = GridWorld(rng=rng)
            frame = env.reset()
            for _ in range(episode_len):
                action = int(rng.integers(0, GAMEGAN_N_ACTIONS))
                next_frame = env.step(action)
                self.triples.append((frame, action, next_frame))
                frame = next_frame

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, i):
        return self.triples[i]

gamegan_dataset = GameGANDataset()
gamegan_loader = DataLoader(gamegan_dataset, batch_size=GAMEGAN_BATCH_SIZE, shuffle=True, drop_last=True)
frame_t, action_t, frame_t1 = next(iter(gamegan_loader))
print(f"GameGAN training triples: {len(gamegan_dataset)}, example actions: "
      f"{[GAMEGAN_ACTION_NAMES[a] for a in action_t[:5].tolist()]}")
show_images(torch.cat([frame_t[:8], frame_t1[:8]]), title="Real transitions: frame_t (top) -> frame_t+1 (bottom)", nrow=8)


### 12.3 Model Architecture

In [ ]:
class GameGAN_Encoder(nn.Module):
    def __init__(self, nc=3, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, nf, 4, 2, 1), nn.LeakyReLU(0.2, True),                              # 32 -> 16
            nn.Conv2d(nf, nf * 2, 4, 2, 1), nn.BatchNorm2d(nf * 2), nn.LeakyReLU(0.2, True),   # 16 -> 8
            nn.Conv2d(nf * 2, nf * 4, 4, 2, 1), nn.BatchNorm2d(nf * 4), nn.LeakyReLU(0.2, True),  # 8 -> 4
        )

    def forward(self, x):
        return self.net(x)


class GameGAN_Decoder(nn.Module):
    def __init__(self, nf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nf * 4, nf * 2, 4, 2, 1), nn.BatchNorm2d(nf * 2), nn.ReLU(True),  # 4 -> 8
            nn.ConvTranspose2d(nf * 2, nf, 4, 2, 1), nn.BatchNorm2d(nf), nn.ReLU(True),          # 8 -> 16
            nn.ConvTranspose2d(nf, nc, 4, 2, 1), nn.Tanh(),                                       # 16 -> 32
        )

    def forward(self, x):
        return self.net(x)


class GameGAN_Generator(nn.Module):
    '''Action-conditioned next-frame predictor with a GRUCell standing in
    for GameGAN's external memory module: `hidden` is threaded through
    calls across a rollout (see 12.6), so predictions can in principle
    depend on more history than just the single previous frame -- the
    minimal recurrent analogue of "remembering game state over time".'''
    def __init__(self, n_actions=GAMEGAN_N_ACTIONS, nf=64, action_embed_dim=16):
        super().__init__()
        self.nf = nf
        self.encoder = GameGAN_Encoder(nf=nf)
        self.action_embed = nn.Embedding(n_actions, action_embed_dim)
        bottleneck_dim = nf * 4 * 4 * 4
        self.hidden_dim = bottleneck_dim
        self.fuse = nn.Linear(bottleneck_dim + action_embed_dim, self.hidden_dim)
        self.gru = nn.GRUCell(self.hidden_dim, self.hidden_dim)
        self.to_bottleneck = nn.Linear(self.hidden_dim, bottleneck_dim)
        self.decoder = GameGAN_Decoder(nf=nf)

    def forward(self, frame, action, hidden=None):
        b = frame.size(0)
        feat = self.encoder(frame).view(b, -1)
        a_emb = self.action_embed(action)
        fused = torch.tanh(self.fuse(torch.cat([feat, a_emb], dim=1)))
        if hidden is None:
            hidden = torch.zeros(b, self.hidden_dim, device=frame.device)
        new_hidden = self.gru(fused, hidden)
        bottleneck = self.to_bottleneck(new_hidden).view(b, self.nf * 4, 4, 4)
        next_frame = self.decoder(bottleneck)
        return next_frame, new_hidden


class GameGAN_Discriminator(nn.Module):
    '''Conditional discriminator judging the TRIPLE (prev_frame, action,
    next_frame) jointly -- exactly the pix2pix/conditional-GAN pattern
    from Section 2, applied to "is this a physically plausible transition"
    instead of "is this the requested class".'''
    def __init__(self, n_actions=GAMEGAN_N_ACTIONS, nf=64, img_size=GAMEGAN_IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.action_embed = nn.Embedding(n_actions, img_size * img_size)
        self.net = nn.Sequential(
            nn.Conv2d(7, nf, 4, 2, 1), nn.LeakyReLU(0.2, True),                               # 3 (prev) + 3 (next) + 1 (action map)
            nn.Conv2d(nf, nf * 2, 4, 2, 1), nn.BatchNorm2d(nf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 2, nf * 4, 4, 2, 1), nn.BatchNorm2d(nf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 4, 1, 4, 1, 0),
        )

    def forward(self, prev_frame, action, next_frame):
        a_map = self.action_embed(action).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([prev_frame, next_frame, a_map], dim=1)
        return self.net(x).view(-1)


G_gamegan = GameGAN_Generator().to(DEVICE)
D_gamegan = GameGAN_Discriminator().to(DEVICE).apply(weights_init)


### 12.4 Loss Functions & Optimizers

In [ ]:
gamegan_adv_criterion = nn.BCEWithLogitsLoss()
gamegan_recon_criterion = nn.L1Loss()
GAMEGAN_LAMBDA_RECON = 10.0   # since we HAVE ground truth next-frames (unlike CycleGAN), a strong direct pixel loss anchors predictions to the true game physics

gamegan_opt_G = torch.optim.Adam(G_gamegan.parameters(), lr=2e-4, betas=(0.5, 0.999))
gamegan_opt_D = torch.optim.Adam(D_gamegan.parameters(), lr=2e-4, betas=(0.5, 0.999))


### 12.5 Training Loop

In [ ]:
gamegan_history = []
for epoch in range(GAMEGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "L1": AvgMeter()}
    pbar = tqdm(gamegan_loader, desc=f"[GameGAN] epoch {epoch+1}/{GAMEGAN_EPOCHS}")
    for frame_t, action_t, frame_t1 in pbar:
        frame_t, action_t, frame_t1 = frame_t.to(DEVICE), action_t.to(DEVICE), frame_t1.to(DEVICE)
        b = frame_t.size(0)

        pred_t1, _ = G_gamegan(frame_t, action_t)   # single-step training: hidden starts fresh each batch (i.i.d. triples)

        # ---- Discriminator ----
        D_gamegan.zero_grad(set_to_none=True)
        logits_real = D_gamegan(frame_t, action_t, frame_t1)
        logits_fake = D_gamegan(frame_t, action_t, pred_t1.detach())
        loss_d = gamegan_adv_criterion(logits_real, torch.ones(b, device=DEVICE)) + \
                 gamegan_adv_criterion(logits_fake, torch.zeros(b, device=DEVICE))
        loss_d.backward()
        gamegan_opt_D.step()

        # ---- Generator: adversarial + strong pixel reconstruction ----
        G_gamegan.zero_grad(set_to_none=True)
        logits_fake_g = D_gamegan(frame_t, action_t, pred_t1)
        loss_g_adv = gamegan_adv_criterion(logits_fake_g, torch.ones(b, device=DEVICE))
        loss_g_recon = gamegan_recon_criterion(pred_t1, frame_t1)
        loss_g = loss_g_adv + GAMEGAN_LAMBDA_RECON * loss_g_recon
        loss_g.backward()
        gamegan_opt_G.step()

        meters["G"].update(loss_g.item()); meters["D"].update(loss_d.item()); meters["L1"].update(loss_g_recon.item())
        pbar.set_postfix(loss_G=f"{loss_g.item():.3f}", loss_D=f"{loss_d.item():.3f}", L1=f"{loss_g_recon.item():.3f}")

    gamegan_history.append((meters["G"].avg, meters["D"].avg, meters["L1"].avg))
    with torch.no_grad():
        G_gamegan.eval()
        pred_demo, _ = G_gamegan(frame_t[:8], action_t[:8])
        G_gamegan.train()
    show_images(torch.cat([frame_t1[:8], pred_demo]), title=f"GameGAN epoch {epoch+1}: real next-frame (top) vs predicted (bottom)", nrow=8)
    save_checkpoint({"G": G_gamegan.state_dict(), "D": D_gamegan.state_dict()}, f"{CHECKPOINT_DIR}/gamegan_epoch{epoch+1}.pt")

print("GameGAN training complete.")


### 12.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def gamegan_rollout(start_frame: torch.Tensor, actions: list) -> list:
    '''The core product function: given a starting frame and a SEQUENCE of
    actions, autoregressively simulate the game -- each predicted frame
    becomes the next step's input, exactly like actually playing the
    (learned, implicit) game engine. `hidden` is threaded across steps.'''
    G_gamegan.eval()
    frame = start_frame.unsqueeze(0).to(DEVICE)
    hidden = None
    frames = [frame.squeeze(0).cpu()]
    for action in actions:
        action_t = torch.tensor([action], device=DEVICE)
        frame, hidden = G_gamegan(frame, action_t, hidden)
        frames.append(frame.squeeze(0).cpu())
    G_gamegan.train()
    return frames

demo_env = GridWorld(rng=np.random.default_rng(0))
start_frame = demo_env.reset()
demo_actions = [0, 0, 3, 3, 1]   # up, up, right, right, down
predicted_frames = gamegan_rollout(start_frame, demo_actions)
show_images(torch.stack(predicted_frames), title=f"GameGAN rollout: {[GAMEGAN_ACTION_NAMES[a] for a in demo_actions]}", nrow=len(predicted_frames))


### 12.7 Visualization & Results

In [ ]:
gs, ds, l1s = zip(*gamegan_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gs, label="G"); axes[0].plot(ds, label="D"); axes[0].legend(); axes[0].set_title("Adversarial losses")
axes[1].plot(l1s); axes[1].set_title("Pixel L1 reconstruction loss")
plt.show()

# Ground-truth rollout vs GENERATED rollout for the SAME action sequence
# and the SAME starting frame -- directly visualizes prediction drift.
eval_env = GridWorld(rng=np.random.default_rng(1))
gt_start = eval_env.reset()
eval_actions = [3, 3, 3, 1, 1, 2, 2, 0]
gt_frames = [gt_start]
for a in eval_actions:
    gt_frames.append(eval_env.step(a))
pred_frames = gamegan_rollout(gt_start, eval_actions)

fig, axes = plt.subplots(2, len(gt_frames), figsize=(1.6 * len(gt_frames), 3.2))
for t in range(len(gt_frames)):
    axes[0, t].imshow(denorm(gt_frames[t]).permute(1, 2, 0).numpy()); axes[0, t].axis("off")
    axes[1, t].imshow(denorm(pred_frames[t]).permute(1, 2, 0).numpy()); axes[1, t].axis("off")
axes[0, 0].set_ylabel("Ground truth", fontsize=9)
axes[1, 0].set_ylabel("GameGAN", fontsize=9)
plt.suptitle("Ground-truth simulator (top) vs learned GameGAN rollout (bottom) -- watch drift accumulate")
plt.tight_layout(); plt.show()

# Quantitative drift: per-step L1 pixel error between ground truth and
# prediction -- error should trend UP with rollout length, illustrating
# the compounding-error problem inherent to autoregressive video prediction.
errors = [F.l1_loss(denorm(pred_frames[t]), denorm(gt_frames[t])).item() for t in range(len(gt_frames))]
plt.figure(figsize=(6, 3)); plt.plot(errors, marker="o")
plt.xlabel("rollout step"); plt.ylabel("L1 error vs ground truth"); plt.title("Compounding prediction drift over a rollout")
plt.show()


### 12.8 User Input → Output Demo

In [ ]:
USER_ACTION_SEQUENCE = ["right", "right", "right", "down", "down"]
RANDOM_SEED = 5

user_env = GridWorld(rng=np.random.default_rng(RANDOM_SEED))
user_start = user_env.reset()
action_ids = [GAMEGAN_ACTION_NAMES.index(a) for a in USER_ACTION_SEQUENCE]
user_rollout = gamegan_rollout(user_start, action_ids)
show_images(torch.stack(user_rollout), title=f"User action sequence: {USER_ACTION_SEQUENCE}", nrow=len(user_rollout))


### 12.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Framing "learn the game" as "predict the next frame
  conditioned on the previous frame and the action" turns environment
  simulation into a conditional-GAN problem almost identical in shape to
  Section 2's cGAN -- the interesting research content is entirely in
  *what* additional structure (memory, disentangled dynamics/rendering)
  makes that prediction remain accurate over long rollouts.
- ⚠️ **Warning.** The compounding-drift plot in 12.7 is the central
  limitation of ALL autoregressive video-prediction models, GameGAN
  included: small per-step errors compound, and after enough steps the
  simulated trajectory can diverge from anything the true environment
  would produce (e.g. "phasing through" a wall the model didn't perfectly
  learn is solid).
- ⚠️ **Warning.** Our GRUCell is a toy stand-in for GameGAN's actual
  external memory module (which explicitly reads/writes a spatial memory
  bank keyed by camera/agent position, enabling correct recall of
  off-screen content) -- do not expect this notebook's model to remember
  anything beyond very recent frames.

**This GAN solved the problem of: learning an implicit simulator of a
(simplified) game environment directly from observed (frame, action,
next-frame) transitions -- predicting future visual game state without
ever being given the game's actual rules.**